In [ ]:
import sys
sys.path.append('../')

from src.run_app import *
from src.prepare_gam import *
from src.utils import *
from gam_rs_utils.binarize_dataset import binarize_dataset
from gam_rs_utils.utils import *
from FasterRisk.src.fasterrisk import fasterrisk
from time import time
import pickle

dataset_settings = {
    'bank': {
        'gap_tolerance': 0.003,
        'starting_num_estimators': 25,
    },
    'compas': {
        'gap_tolerance': 0.002,
        'starting_num_estimators': 25,
    },
    'diabetes': {
        'gap_tolerance': 0.006,
        'starting_num_estimators': 200,
    },
    'heloc_original': {
        'gap_tolerance': 0.002,
        'starting_num_estimators': 25,
    },
    # 'hiv': { # not used
    'netherlands': {
        'gap_tolerance': 0.0005,
        'starting_num_estimators': 20,
    },
    'spambase': {
        'gap_tolerance': 0.0045,
        'starting_num_estimators': 20,
    },
}

tries = 6
results = []
for dataset_name, settings in dataset_settings.items():
    gt = settings['gap_tolerance']

    path = '../datasets/{}.csv'.format(dataset_name)
    dataset = pd.read_csv(path)
    print(f"Dataset: {dataset_name}")
    print(f"Binarized shape: {dataset.shape}")

    for i in range(1, tries + 1):
        ne = settings["starting_num_estimators"] * i

        df, thresholds, header, threshold_guess_time = binarize_dataset(dataset, ne)
        X, y = df.iloc[:, :-1], df.iloc[:, -1]
        header = pd.Index(["intercept"] + list(X.columns)).astype("object")
        X_one_hot, y = utils.get_X_y(X, y)

        start = time()
        rs = fasterrisk.RiskScoreOptimizer(X_one_hot, y, k=10, lb=-100, ub=100, gap_tolerance=gt, select_top_m=-1, maxAttempts=25)
        rs.optimize_with_swaps(swaps=3, fanout_decay=1, feature_selection="top")
        end = time()

        result = {
            "dataset": dataset_name,
            "dataset_shape": dataset.shape,
            "num_estimators": ne,
            "gap_tolerance": gt,
            "feature_selection": "top",
            "threshold_guess_time": threshold_guess_time,
            "num_features": len(header),
            "runtime": end - start,
            "betas": rs.sparseDiversePool_betas,
            "beta0": rs.sparseDiversePool_beta0,
            "num_solutions": rs.sparseDiversePool_betas.shape[0],
            "loss": get_loss(X_one_hot, y, rs.sparseDiversePool_beta0, rs.sparseDiversePool_betas)
        }
        results.append(result)

        print(f"\t{len(header)} features, {result['num_solutions']} solutions, {result['runtime']:.2f} seconds")

with open("results/features.pkl", "wb") as f:
    pickle.dump(results, f)

Dataset: bank
Binarized shape: (4521, 17)
	16 features, 64 solutions, 34.25 seconds
	25 features, 0 solutions, 18.48 seconds
	30 features, 0 solutions, 16.15 seconds
	38 features, 1234 solutions, 168.40 seconds
	42 features, 761 solutions, 123.33 seconds
	44 features, 827 solutions, 128.67 seconds
Dataset: compas
Binarized shape: (6907, 8)
	14 features, 5 solutions, 11.60 seconds
	22 features, 126 solutions, 33.31 seconds
	28 features, 267 solutions, 72.21 seconds
	29 features, 403 solutions, 94.71 seconds
	33 features, 651 solutions, 165.82 seconds
	34 features, 654 solutions, 168.28 seconds
Dataset: diabetes
Binarized shape: (768, 9)
	50 features, 118 solutions, 8.15 seconds
	69 features, 132 solutions, 9.55 seconds
	82 features, 204 solutions, 11.18 seconds
	92 features, 204 solutions, 11.37 seconds
	98 features, 238 solutions, 12.15 seconds
	109 features, 238 solutions, 12.51 seconds
Dataset: heloc_original
Binarized shape: (10459, 24)
	18 features, 43 solutions, 60.67 seconds
	29 

FileNotFoundError: [Errno 2] No such file or directory: 'results/features.pkl'